## Section 1 — install packages
Same as Assignment 1, with one addition: `openslide-python` and `requests` for the GDC tile API.

- `openslide-python` reads whole-slide images (SVS/TIFF); here we use it to validate downloaded tiles.
- `requests` is used to call the GDC REST API and cBioPortal API without needing any credentials.
- All other packages are identical to the colorectal notebook.


In [ ]:
# ============================================================
# 1A) INSTALL SYSTEM DEPENDENCY (OpenSlide native library)
# ============================================================
import subprocess, sys
subprocess.check_call(["apt-get", "install", "-y", "-q", "openslide-tools"], stdout=subprocess.DEVNULL)

# ============================================================
# 1B) INSTALL PYTHON PACKAGES
# ============================================================
import importlib.util

PACKAGE_TO_MODULE = {
    "timm":            "timm",
    "captum":          "captum",
    "shap":            "shap",
    "scikit-image":    "skimage",
    "lime":            "lime",
    "openslide-python":"openslide",
    "requests":        "requests",
}

missing = [pkg for pkg, mod in PACKAGE_TO_MODULE.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All packages already installed.")

In [ ]:
# Fix NumPy / pandas binary incompatibility in Colab
!pip install -q --force-reinstall --no-cache-dir \
    "numpy==1.26.4" \
    "pandas==2.2.2" \
    "numba==0.60.0" \
    "shap==0.46.0"


## Section 2 — imports, seed, device, and settings
Nearly identical to Assignment 1.

Key differences:
- `tensorflow-datasets` is removed (we load TCGA-BRCA directly, not via TFDS).
- `requests` and `os` are used more extensively for API calls.
- `CLASS_NAMES` now reflects the four breast cancer molecular subtypes.
- `CONFIG` has new keys for the TCGA data pipeline.

Label mapping (receptor-based, matching TCGA clinical data):
| ID | Class | Definition |
|----|-------|-----------|
| 0 | Luminal A | ER+ or PR+, HER2-, PAM50 LumA |
| 1 | Luminal B | ER+ or PR+, HER2+ **or** grade 3 |
| 2 | HER2-enriched | ER-, PR-, HER2+ |
| 3 | Triple Negative | ER-, PR-, HER2- |


In [ ]:
# ============================================================
# 2) IMPORTS, SEED, DEVICE, AND GLOBAL SETTINGS
# ============================================================
import os, gc, math, time, random, warnings, requests, zipfile, shutil
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, log_loss, roc_auc_score,
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import LogisticRegression

from skimage import color, exposure, filters, feature, measure, morphology, segmentation, transform, util
from scipy import ndimage as ndi

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

import timm
import shap
from captum.attr import Saliency, IntegratedGradients, LayerGradCam, Occlusion

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_RUN = True   # Set to False for full training

# ── Device ────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ── Class names ───────────────────────────────────────────
# Four molecular subtypes of breast cancer (receptor-based)
CLASS_NAMES = ["Luminal_A", "Luminal_B", "HER2_Enriched", "Triple_Negative"]
NUM_CLASSES = len(CLASS_NAMES)

# ── Global CONFIG ─────────────────────────────────────────
CONFIG = {
    # Image settings
    "image_size":    224,
    "patch_size":    256,          # Tile size downloaded from GDC (pixels)
    "batch_size":    32,
    "num_workers":   2,

    # FAST_RUN caps (patches per subtype)
    "max_patches_per_class_fast": 80,   # ~320 total images — light enough for free Colab GPU
    "max_patches_per_class_full": 400,  # ~1 600 total

    # Training epochs
    "epochs_resnet": 3  if FAST_RUN else 10,
    "epochs_vit":    3  if FAST_RUN else 8,
    "epochs_fusion": 5  if FAST_RUN else 20,

    # Learning rates
    "lr_resnet":     3e-4,
    "lr_vit":        2e-4,
    "lr_fusion":     1e-3,
    "weight_decay":  1e-4,

    # Other
    "run_lime":      False,
    "num_classes":   NUM_CLASSES,
}

# FAST_RUN convenience alias
MAX_PER_CLASS = (CONFIG["max_patches_per_class_fast"] if FAST_RUN
                 else CONFIG["max_patches_per_class_full"])

print(f"FAST_RUN={FAST_RUN} | patches per class: {MAX_PER_CLASS} | classes: {CLASS_NAMES}")


## Section 3 — load the TCGA-BRCA dataset
This section replaces the TFDS `colorectal_histology` loader.

**Two data sources are combined:**

1. **Clinical / molecular table** — downloaded from cBioPortal (TCGA Breast PanCancer Atlas 2018).
   Contains PAM50 subtype, ER/PR/HER2 IHC status, grade, stage, age — all free, no login.

2. **Diagnostic H&E image tiles** — downloaded from the NCI Genomic Data Commons (GDC) REST API.
   The GDC tile endpoint (`/tile/{file_id}?level=&x=&y=`) returns individual 256 × 256 pixel JPEG tiles
   from a whole-slide image **without** downloading the full SVS file (~1–3 GB each).
   No login or `gdc-client` is needed for open-access data.

**Pipeline:**
```
cBioPortal CSV  ──►  label each case (Luminal A / B / HER2-E / TNBC)
GDC API        ──►  list diagnostic slide file IDs per case
GDC tile API   ──►  download N tiles per slide  ──►  images[]
Merge          ──►  images[], labels[], clinical_df[]
```


In [ ]:
# ============================================================
# 3A) DOWNLOAD CLINICAL DATA FROM cBIOPORTAL
# ============================================================
# TCGA Breast Cancer PanCancer Atlas 2018
# Direct download — no authentication required.

CBIOPORTAL_BASE = (
    "https://cbioportal-datahub.s3.amazonaws.com/brca_tcga_pan_can_atlas_2018.tar.gz"
)
DATA_DIR = Path("tcga_brca_data")
DATA_DIR.mkdir(exist_ok=True)

tarball = DATA_DIR / "brca_tcga_pan_can_atlas_2018.tar.gz"
if not tarball.exists():
    print("Downloading cBioPortal clinical archive …")
    r = requests.get(CBIOPORTAL_BASE, stream=True, timeout=120)
    r.raise_for_status()
    with open(tarball, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    print("  Done:", tarball.stat().st_size // (1 << 20), "MB")

# Extract
import tarfile
with tarfile.open(tarball, "r:gz") as tar:
    tar.extractall(DATA_DIR)

# Locate the clinical patient file
clin_files = list(DATA_DIR.rglob("data_clinical_patient.txt"))
print("Clinical file:", clin_files[0])


In [ ]:
# ============================================================
# 3B) PARSE CLINICAL TABLE AND ASSIGN LABELS
# ============================================================

clin_raw = pd.read_csv(clin_files[0], sep="\t", comment="#", low_memory=False)

# Keep columns we need
KEEP_COLS = [
    "PATIENT_ID",
    "AGE",
    "AJCC_PATHOLOGIC_TUMOR_STAGE",
    "TUMOR_GRADE",
    "ER_STATUS_BY_IHC",
    "PR_STATUS_BY_IHC",
    "IHC_HER2",
    "ONCOTREE_CODE",
]
available = [c for c in KEEP_COLS if c in clin_raw.columns]
clin = clin_raw[available].copy()

# PAM50 subtype comes from a separate molecular file in the same archive
pam50_files = list(DATA_DIR.rglob("data_mrna_seq_v2_rsem_zscores_ref_diploid_samples.txt"))
# Subtype assignments may also be in a supplemental file; use IHC-based labelling as fallback.

subtypes_file = list(DATA_DIR.rglob("*subtype*"))
if subtypes_file:
    sub_df = pd.read_csv(subtypes_file[0], sep="\t", comment="#", low_memory=False)
    print("Subtype file found:", subtypes_file[0])
else:
    print("No subtype file found; using IHC-based label assignment.")
    sub_df = None

def assign_label_ihc(row):
    """
    Receptor-based subtype assignment (standard clinical definition).
    Returns class ID (int) or -1 for unclassifiable.
    """
    er  = str(row.get("ER_STATUS_BY_IHC",  "")).strip().upper()
    pr  = str(row.get("PR_STATUS_BY_IHC",  "")).strip().upper()
    her = str(row.get("IHC_HER2",          "")).strip().upper()
    grade = str(row.get("TUMOR_GRADE",     "")).strip()

    hr_pos = ("POSITIVE" in er) or ("POSITIVE" in pr)
    her2_pos = "POSITIVE" in her

    if hr_pos and not her2_pos:
        # Grade 3 → Luminal B; otherwise Luminal A
        return 1 if "3" in grade else 0
    elif hr_pos and her2_pos:
        return 1   # Luminal B (HER2+)
    elif not hr_pos and her2_pos:
        return 2   # HER2-enriched
    elif not hr_pos and not her2_pos:
        return 3   # Triple Negative
    return -1      # Unknown

clin["label"] = clin.apply(assign_label_ihc, axis=1)
clin = clin[clin["label"] >= 0].reset_index(drop=True)

# Convert PATIENT_ID to TCGA format (e.g. TCGA-A1-A0SB)
clin["patient_id"] = clin["PATIENT_ID"].str.upper().str.strip()

label_counts = clin["label"].value_counts().sort_index()
print("\nLabel distribution (before image download):")
for lid, name in enumerate(CLASS_NAMES):
    print(f"  {lid} {name}: {label_counts.get(lid, 0)} patients")


In [ ]:
# ============================================================
# 3C) QUERY GDC API FOR DIAGNOSTIC SLIDE FILE IDs
# ============================================================
# GDC API endpoint — open access, no login required.
GDC_FILES_EP  = "https://api.gdc.cancer.gov/files"
GDC_TILE_EP   = "https://api.gdc.cancer.gov/tile"

def gdc_get_slide_file_ids(case_ids, batch=100):
    """
    Given a list of TCGA case IDs, return a dict  {case_id: [file_id, …]}
    for diagnostic FFPE whole-slide SVS files.
    """
    result = {}
    for start in range(0, len(case_ids), batch):
        chunk = case_ids[start : start + batch]
        payload = {
            "filters": {
                "op": "and",
                "content": [
                    {"op": "in",  "content": {"field": "cases.submitter_id", "value": chunk}},
                    {"op": "=",   "content": {"field": "data_type",          "value": "Slide Image"}},
                    {"op": "=",   "content": {"field": "data_format",        "value": "SVS"}},
                    {"op": "=",   "content": {"field": "experimental_strategy","value": "Diagnostic Slide"}},
                    {"op": "=",   "content": {"field": "access",             "value": "open"}},
                ]
            },
            "fields": "file_id,cases.submitter_id",
            "size": batch * 5,
        }
        r = requests.post(GDC_FILES_EP, json=payload, timeout=60)
        r.raise_for_status()
        for hit in r.json()["data"]["hits"]:
            fid = hit["file_id"]
            for c in hit.get("cases", []):
                cid = c["submitter_id"].upper()
                result.setdefault(cid, []).append(fid)
    return result

# Limit to a balanced subset for download speed
N_CASES_PER_CLASS = 25 if FAST_RUN else 100
case_pool = {}
for lid in range(NUM_CLASSES):
    subset = clin[clin["label"] == lid]["patient_id"].tolist()
    random.shuffle(subset)
    case_pool[lid] = subset[:N_CASES_PER_CLASS]

all_case_ids = [cid for ids in case_pool.values() for cid in ids]
print(f"Querying GDC for {len(all_case_ids)} cases …")
slide_map = gdc_get_slide_file_ids(all_case_ids)
print(f"Found slides for {len(slide_map)} / {len(all_case_ids)} cases.")


In [ ]:
# ============================================================
# 3D) DOWNLOAD IMAGE TILES FROM GDC TILE API
# ============================================================
# GDC provides JPEG tiles without downloading the full SVS.
# Endpoint: GET /tile/{file_id}?level={level}&x={x}&y={y}
#
# level=7 gives ~256×256 px tiles at moderate magnification (~5×).
# For higher magnification increase level (smaller tiles, more detail).

TILE_DIR  = DATA_DIR / "tiles"
TILE_DIR.mkdir(exist_ok=True)

TILES_PER_SLIDE = 4 if FAST_RUN else 8
LEVEL           = 7    # 7 ≈ 5× magnification; raise to 8–9 for 10–20×

def download_tile(file_id, level, x, y, save_path):
    if save_path.exists():
        return True
    url = f"{GDC_TILE_EP}/{file_id}?level={level}&x={x}&y={y}"
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and len(r.content) > 500:
            save_path.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

# Sample tile coordinates to fetch (central region of the slide)
SAMPLE_COORDS = [(x, y) for x in range(2, 6) for y in range(2, 6)]
random.shuffle(SAMPLE_COORDS)
SAMPLE_COORDS = SAMPLE_COORDS[:TILES_PER_SLIDE]

images_raw, labels_raw, meta_rows = [], [], []

for lid in range(NUM_CLASSES):
    for case_id in tqdm(case_pool[lid], desc=f"Class {CLASS_NAMES[lid]}", leave=False):
        file_ids = slide_map.get(case_id, [])
        if not file_ids:
            continue
        fid = file_ids[0]   # use first slide per case
        collected = 0
        for (tx, ty) in SAMPLE_COORDS:
            if collected >= TILES_PER_SLIDE:
                break
            fname = TILE_DIR / f"{fid}_{level}_{tx}_{ty}.jpg"
            ok = download_tile(fid, level, tx, ty, fname)
            if not ok:
                continue
            try:
                img = np.array(Image.open(fname).convert("RGB").resize((256, 256)))
                if img.mean() > 240 or img.mean() < 10:
                    continue   # skip near-white (glass) or near-black tiles
                images_raw.append(img)
                labels_raw.append(lid)
                row = clin[clin["patient_id"] == case_id].iloc[0].to_dict()
                row["label"] = lid
                row["file_id"] = fid
                meta_rows.append(row)
                collected += 1
            except Exception:
                continue

print(f"\nTotal tiles downloaded: {len(images_raw)}")
for lid, name in enumerate(CLASS_NAMES):
    n = sum(1 for l in labels_raw if l == lid)
    print(f"  {name}: {n} tiles")


In [ ]:
# ============================================================
# 3E) BALANCE CLASSES AND FINALISE ARRAYS
# ============================================================

images_raw  = np.array(images_raw,  dtype=np.uint8)
labels_raw  = np.array(labels_raw,  dtype=np.int64)
meta_df_raw = pd.DataFrame(meta_rows).reset_index(drop=True)

# Balance to MAX_PER_CLASS tiles per subtype
balanced_idx = []
for lid in range(NUM_CLASSES):
    idx = np.where(labels_raw == lid)[0]
    np.random.shuffle(idx)
    balanced_idx.extend(idx[:MAX_PER_CLASS].tolist())
balanced_idx = np.array(sorted(balanced_idx))

images     = images_raw[balanced_idx]
labels     = labels_raw[balanced_idx]
meta_df    = meta_df_raw.iloc[balanced_idx].reset_index(drop=True)
class_names = CLASS_NAMES

print("Final dataset shape:", images.shape)
print("Labels:", dict(zip(*np.unique(labels, return_counts=True))))

# ── Attach numeric clinical features to meta_df ───────────────────────────────
# These will be incorporated as extra tabular columns in Section 6.
def encode_stage(s):
    s = str(s).upper()
    if "IV" in s: return 4
    if "III" in s: return 3
    if "II" in s: return 2
    if "I" in s: return 1
    return np.nan

def encode_grade(g):
    g = str(g)
    for c in ["1","2","3"]:
        if c in g: return int(c)
    return np.nan

meta_df["stage_num"] = meta_df.get("AJCC_PATHOLOGIC_TUMOR_STAGE", pd.Series(dtype=str)).apply(encode_stage)
meta_df["grade_num"] = meta_df.get("TUMOR_GRADE", pd.Series(dtype=str)).apply(encode_grade)
meta_df["age_num"]   = pd.to_numeric(meta_df.get("AGE", pd.Series(dtype=str)), errors="coerce")

CLINICAL_FEATURE_COLS = ["stage_num", "grade_num", "age_num"]
print("Clinical columns added:", CLINICAL_FEATURE_COLS)


## Section 4 — basic exploration
Same as Assignment 1 — inspect class balance and visualise representative tiles.

Breast cancer patches look different from colorectal histology:
- **Luminal A**: low-grade glands, sparse nuclei, abundant stroma
- **Luminal B**: moderate pleomorphism, denser glands
- **HER2-enriched**: high nuclear grade, solid growth patterns
- **TNBC**: pushing borders, lymphocytic infiltration, necrosis


In [ ]:
# ============================================================
# 4) BASIC EXPLORATION
# ============================================================

label_counts = pd.Series(labels).value_counts().sort_index()
label_df = pd.DataFrame({
    "class_id":   list(range(len(class_names))),
    "class_name": class_names,
    "count":      label_counts.reindex(range(len(class_names))).values,
})
display(label_df)

plt.figure(figsize=(10, 4))
plt.bar(label_df["class_name"], label_df["count"],
        color=["#2196F3","#64B5F6","#FF5722","#9C27B0"])
plt.title("TCGA-BRCA tile class distribution")
plt.ylabel("Number of tiles")
plt.xticks(rotation=15, ha="right")
plt.tight_layout(); plt.show()

def show_class_examples(images, labels, class_names, n_per_class=4):
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, n_per_class,
                             figsize=(3 * n_per_class, 3 * n_classes))
    for class_id in range(n_classes):
        idx = np.where(labels == class_id)[0]
        chosen = np.random.choice(idx, size=min(n_per_class, len(idx)), replace=False)
        for j in range(n_per_class):
            ax = axes[class_id, j]
            ax.axis("off")
            if j < len(chosen):
                ax.imshow(images[chosen[j]])
                if j == 0:
                    ax.set_title(class_names[class_id], fontsize=9)
    plt.suptitle("Sample H&E tiles — TCGA-BRCA", y=1.01, fontsize=13)
    plt.tight_layout(); plt.show()

show_class_examples(images, labels, class_names, n_per_class=4)


## Section 5 — classical segmentation and detection
**Identical pipeline to Assignment 1** — HED colour deconvolution works on any H&E slide.

Breast cancer tiles differ in nuclei density and architecture by subtype, so the
morphology features extracted in Section 6 will carry real biological signal.
No changes to the segmentation code are needed.


In [ ]:
# ============================================================
# 5) CLASSICAL SEGMENTATION AND DETECTION
# (Code is identical to Assignment 1 — paste your Section 5 here)
# ============================================================

def normalize_01(x):
    x = x.astype(np.float32)
    x -= x.min(); x /= (x.max() + 1e-8)
    return x

def segment_nuclei_from_histology(img_rgb, min_size=16):
    hed = color.rgb2hed(img_rgb)
    hematoxylin = normalize_01(-hed[:, :, 0])
    smooth = filters.gaussian(hematoxylin, sigma=1.0)
    thresh = filters.threshold_otsu(smooth)
    mask = smooth > thresh
    mask = morphology.remove_small_objects(mask, min_size=min_size)
    mask = morphology.remove_small_holes(mask, area_threshold=min_size)
    mask = morphology.binary_opening(mask,  morphology.disk(1))
    mask = morphology.binary_closing(mask,  morphology.disk(1))

    distance = ndi.distance_transform_edt(mask)
    if mask.sum() == 0:
        return hematoxylin, smooth, mask.astype(np.uint8), np.zeros_like(mask, dtype=np.int32)

    coords = feature.peak_local_max(distance, footprint=np.ones((7,7), dtype=bool), labels=mask)
    marker_mask = np.zeros_like(mask, dtype=bool)
    if len(coords):
        marker_mask[tuple(coords.T)] = True
    else:
        marker_mask = mask.copy()

    markers, _ = ndi.label(marker_mask)
    labels_ws   = segmentation.watershed(-distance, markers, mask=mask)
    return hematoxylin, smooth, mask.astype(np.uint8), labels_ws

def detect_nuclei_boxes(labels_ws):
    boxes = []
    for region in measure.regionprops(labels_ws):
        minr, minc, maxr, maxc = region.bbox
        boxes.append({"minr": minr, "minc": minc, "maxr": maxr, "maxc": maxc,
                      "area": region.area, "eccentricity": region.eccentricity})
    return boxes

# Quick visual check on one tile from each class
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(10, 3.5 * NUM_CLASSES))
for lid in range(NUM_CLASSES):
    idx = np.where(labels == lid)[0][0]
    img = images[idx]
    hem, smooth, mask, labels_ws = segment_nuclei_from_histology(img)
    axes[lid, 0].imshow(img);            axes[lid, 0].set_title(f"{CLASS_NAMES[lid]}\nOriginal", fontsize=8)
    axes[lid, 1].imshow(hem, cmap="Blues"); axes[lid, 1].set_title("Hematoxylin", fontsize=8)
    axes[lid, 2].imshow(mask, cmap="gray"); axes[lid, 2].set_title("Nucleus mask", fontsize=8)
    for ax in axes[lid]: ax.axis("off")
plt.tight_layout(); plt.show()


## Section 6 — handcrafted tabular feature extraction
**Extended version of Assignment 1 Section 6.**

The morphology pipeline (HED stats, GLCM, LBP, nucleus count/area/eccentricity)
is kept exactly as-is — it applies directly to breast H&E tiles.

**New in this section:** after morphology extraction we append the three
**clinical features** (stage, grade, age) from `meta_df`.
This makes the tabular modality genuinely multimodal — image-derived morphology
**plus** patient-level clinical context.


In [ ]:
# ============================================================
# 6) HANDCRAFTED TABULAR FEATURE EXTRACTION
# (Morphology pipeline identical to Assignment 1 + clinical append)
# ============================================================

def safe_stats(values):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return {"mean":0.,"std":0.,"min":0.,"max":0.,"median":0.}
    return {"mean":float(np.mean(values)),"std":float(np.std(values)),
            "min":float(np.min(values)),"max":float(np.max(values)),
            "median":float(np.median(values))}

def glcm_features(gray_u8, distances=(1,2),
                  angles=(0, np.pi/4, np.pi/2, 3*np.pi/4), levels=32):
    gray_small = transform.resize(gray_u8,(64,64),preserve_range=True,anti_aliasing=True).astype(np.uint8)
    quantized  = np.clip(np.floor(gray_small.astype(np.float32)/(256/levels)).astype(np.uint8), 0, levels-1)
    glcm = feature.graycomatrix(quantized, list(distances), list(angles),
                                levels=levels, symmetric=True, normed=True)
    return {f"glcm_{p}": float(feature.graycoprops(glcm, p).mean())
            for p in ["contrast","dissimilarity","homogeneity","energy","correlation","ASM"]}

def lbp_histogram(gray_u8, points=8, radius=1):
    gray_small = transform.resize(gray_u8,(64,64),preserve_range=True,anti_aliasing=True).astype(np.uint8)
    lbp  = feature.local_binary_pattern(gray_small, P=points, R=radius, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=int(points+2), range=(0,int(points+2)),density=True)
    return {f"lbp_{i}": float(hist[i]) for i in range(len(hist))}

def extract_features_from_image(img_rgb):
    feats = {}
    # ── Colour statistics per channel (RGB + HED) ────────────────────────────
    for ch, name in enumerate(["R","G","B"]):
        ch_vals = img_rgb[:,:,ch].astype(np.float32)
        feats.update({f"rgb_{name}_{k}": v for k, v in safe_stats(ch_vals.ravel()).items()})
    hed = color.rgb2hed(img_rgb)
    for ch, name in enumerate(["H","E","D"]):
        feats.update({f"hed_{name}_{k}": v for k, v in safe_stats(hed[:,:,ch].ravel()).items()})
    # ── Texture (GLCM + LBP) ─────────────────────────────────────────────────
    gray_u8 = (color.rgb2gray(img_rgb) * 255).astype(np.uint8)
    feats.update(glcm_features(gray_u8))
    feats.update(lbp_histogram(gray_u8))
    # ── Nucleus segmentation morphology ──────────────────────────────────────
    hem, smooth, mask, labels_ws = segment_nuclei_from_histology(img_rgb)
    regions = measure.regionprops(labels_ws)
    boxes   = detect_nuclei_boxes(labels_ws)
    feats["nuclei_count"]       = len(regions)
    feats["nuclei_pixel_ratio"] = float(mask.sum()) / float(mask.size)
    for prop in ["area","eccentricity","perimeter","solidity"]:
        vals = [getattr(r, prop) for r in regions if hasattr(r, prop)]
        feats.update({f"nucleus_{prop}_{k}": v for k, v in safe_stats(vals).items()})
    box_areas = [(b["maxr"]-b["minr"])*(b["maxc"]-b["minc"]) for b in boxes]
    feats["box_count"]    = len(boxes)
    feats.update({f"box_area_{k}": v for k, v in safe_stats(box_areas).items()})
    return feats

print(f"Extracting features from {len(images)} tiles …")
all_feats = []
for i in tqdm(range(len(images))):
    f = extract_features_from_image(images[i])
    f["index"] = i
    f["label"] = int(labels[i])
    all_feats.append(f)

feature_df = pd.DataFrame(all_feats)

# ── Append clinical features ──────────────────────────────────────────────────
for col in CLINICAL_FEATURE_COLS:
    if col in meta_df.columns:
        feature_df[col] = meta_df[col].values

# Median-impute any NaNs
feature_df = feature_df.fillna(feature_df.median(numeric_only=True))
print("Feature table shape:", feature_df.shape)
display(feature_df.head(3))


## Section 7 — inspect the tabular modality
Identical to Assignment 1. Check that clinical features (stage_num, grade_num, age_num)
show meaningful variation across subtypes.

Expected patterns:
- **TNBC**: higher grade, younger age at diagnosis
- **Luminal A**: lower grade, older age, earlier stage
- **HER2-enriched**: intermediate grade, high stage


In [ ]:
# ============================================================
# 7) INSPECT THE TABULAR MODALITY
# (Identical to Assignment 1 — added clinical features will appear automatically)
# ============================================================

feature_columns = [c for c in feature_df.columns if c not in ["index","label"]]
display(feature_df[feature_columns].describe().T.head(20))

# Plot clinical features by subtype
clin_plot_cols = [c for c in CLINICAL_FEATURE_COLS if c in feature_df.columns]
morph_plot_cols = [c for c in ["nuclei_count","nuclei_pixel_ratio","nucleus_area_mean","glcm_contrast"]
                   if c in feature_df.columns]
plot_cols = clin_plot_cols + morph_plot_cols

if plot_cols:
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(5*len(plot_cols), 4))
    if len(plot_cols) == 1: axes = [axes]
    for ax, col in zip(axes, plot_cols):
        grouped = [feature_df.loc[feature_df["label"]==lid, col].dropna().values
                   for lid in range(NUM_CLASSES)]
        ax.boxplot(grouped, labels=[n[:6] for n in class_names], showfliers=False)
        ax.set_title(col, fontsize=9)
        ax.tick_params(axis="x", rotation=30)
    plt.suptitle("Feature distributions by subtype", fontsize=12)
    plt.tight_layout(); plt.show()


## Section 8 — train, validation, and test splits
Identical to Assignment 1. Stratified split keeps subtype proportions balanced.


In [ ]:
# ============================================================
# 8) TRAIN / VALIDATION / TEST SPLITS
# (Identical to Assignment 1)
# ============================================================

all_indices = np.arange(len(images))
train_val_idx, test_idx, y_train_val, y_test = train_test_split(
    all_indices, labels, test_size=0.20, random_state=SEED, stratify=labels)
train_idx, val_idx, y_train, y_val = train_test_split(
    train_val_idx, y_train_val, test_size=0.20, random_state=SEED, stratify=y_train_val)

print("Train:", len(train_idx), "| Val:", len(val_idx), "| Test:", len(test_idx))

# Tabular arrays (X_tab)
feature_columns = [c for c in feature_df.columns if c not in ["index","label"]]
X_tab = feature_df[feature_columns].values.astype(np.float32)

scaler = StandardScaler().fit(X_tab[train_idx])
X_tab_train = scaler.transform(X_tab[train_idx])
X_tab_val   = scaler.transform(X_tab[val_idx])
X_tab_test  = scaler.transform(X_tab[test_idx])

print("Tabular feature matrix shape:", X_tab.shape)
print("Feature columns:", feature_columns[:5], "…")


## Sections 9 – 28 — identical to Assignment 1
Everything from Section 9 onward (metrics helpers, classical baselines,
PyTorch dataloaders, ResNet18, ViT-tiny, MobileNetV3-Small, fusion models,
interpretability, error analysis, LIME, case study) **runs without modification**.

Copy the corresponding cells from `Amal_Assignment1.ipynb` directly here.
The only variable names that changed are:

| Assignment 1 | This notebook |
|---|---|
| `colorectal_histology` (8 classes) | TCGA-BRCA (4 classes) |
| `class_names` | `CLASS_NAMES` (already set above) |
| `num_classes` | `NUM_CLASSES = 4` (already set) |
| No clinical features | `stage_num`, `grade_num`, `age_num` in `feature_df` |

Everything else — model definitions, training loops, calibration, SHAP,
GradCAM, fusion — is architecture-agnostic and works for any number of classes ≥ 2.


In [ ]:
# ============================================================
# 9 – 28) PASTE YOUR ASSIGNMENT 1 CODE HERE
# ============================================================
# Start from the cell labelled:
#   "Section 9 - metrics and calibration helpers"
# and copy every cell to the end of the notebook.
#
# The four class label strings in confusion matrix plots will
# automatically update because they read from `class_names`.
#
# One small fix needed in Section 4-style show_class_examples calls:
#   replace  n_per_class=4  with  n_per_class=4  (no change needed).
#
# In the Section 28 ordinal head example, num_grades=4 instead of 8.
print("Sections 9–28: copy from Assignment 1 notebook — no code changes required.")


## Save results


In [ ]:
results_df.to_csv("tcga_brca_results.csv", index=False)
display(results_df)
print("Pipeline complete.")
